In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import gc

from fp_constants import *

# Find the repo root and add it to sys.path FIRST
repo = Path.cwd()
while not (repo / "functions").exists() and repo.parent != repo:
    repo = repo.parent
sys.path[:0] = [str(repo), str(Path.cwd().parent.parent)]

import tempfile, py7zr
import geopandas as gpd
import glob, json
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from rasterio.warp import reproject, calculate_default_transform

from constants import *
from functions.plot_hillshade import plot_hillshade
from utils import *
from fp_plotting import *
from ecosystems.cs.cs_plotting import *
from ecosystems.cs.cs_data_process import *


FP_PATH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/composite_priority_2020COFAP_unpacked/commondata/raster_data/comp_121_int3")

### PLOT Forest Priority LAYER ###

In [ ]:


# Reproject priority raster to match your hillshade/counties CRS
pri_arr, pri_extent = reproject_raster(FP_PATH, TARGET_CRS)

# Mask background — 0 is nodata for this raster, also catch any float sentinels/NaN
pri_arr = np.ma.masked_where(
    (pri_arr == 0) | (pri_arr == -3.4028234663852886e+38) | np.isnan(pri_arr),
    pri_arr,
)

step = 2
pri_small = pri_arr[::step, ::step]

cmap = plt.get_cmap("Spectral_r")
norm = Normalize(vmin=float(pri_small.min()), vmax=float(pri_small.max()))

# Hillshade base
fig, ax = plot_hillshade()

# Priority layer on top
im = ax.imshow(
    pri_small, extent=pri_extent, cmap=cmap, norm=norm,
    interpolation="nearest", alpha=0.75, zorder=2,
)
cbar = fig.colorbar(im, ax=ax, shrink=0.6, label="Composite Priority (low → high)")
cbar.set_ticks([])

# Counties on top
plt_counties(COUNTIES_PATH, county_edgecolor="black", county_linewidth=1.0, ax=ax)

plt.tight_layout()
plt.show()

### PLOT Bivariate ###

In [ ]:
# ============================================================
# Pipeline
# ============================================================

LAYERS_TO_PLOT = ["5 day max precip", "drought__D3", "Days Exceeding 95th Percentile Maximum Temperature"] #["drought__D3"] #["5 day max precip"]
#, "Days Exceeding 95th Percentile Maximum Temperature"
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/fp_bivariate")

# CS = fine reference grid — build once
cs_transform, cs_width, cs_height = build_reference_grid(str(FP_PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(FP_PATH), cs_transform, cs_width, cs_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(cs_transform, cs_width, cs_height)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue

    for fp in files:
        # Upsample climate to the CS grid
        clim_aligned = reproject_to_grid(
            fp, cs_transform, cs_width, cs_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        # Classify
        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        # Save everything
        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        save_bivariate_outputs(
            run_dir,
            bivar_map=codes_2d,
            extent=extent,
            color_dict=layers[layer_name].get("bivar"),
            x_label="Composite Forest Priority →",
            y_label=f"{cfg.get('label', layer_name)} →",
            n_classes=N_CLASSES,
            x_breaks=x_breaks,
            y_breaks=y_breaks,
            meta_extra={
                "layer_name": layer_name,
                "climate_file": str(fp),
                "cs_file": str(FP_PATH),
                "x_variable": "Composite Forest Priority",
                "y_variable": cfg.get("label", layer_name),
                "y_units": cfg.get("cbar_label", ""),
                "classification_method": "quantile",
            },
        )
        print(f"✓ saved {run_dir.name}/")

### Plot Vegetation ###

In [ ]:
def plot_veg_composition(
    bivar_map,
    veg_aligned,
    bivar_colors,
    hazard_label="Hazard",
    title=None,
    veg_labels=None,
    sort_by="size",
    figsize=None,
):
    """
    Stacked horizontal bar chart showing the nonant composition of each
    vegetation class, with a companion total-pixels bar and 3x3 legend.

    Parameters
    ----------
    bivar_map : 2D np.ndarray
        Bivariate classification (0–8, -1 for NA). Same shape as veg_aligned.
    veg_aligned : 2D np.ndarray
        Vegetation class raster aligned to the same grid. NaN for invalid.
    bivar_colors : dict {int -> color}
        9-class bivariate color mapping (keys 0–8).
    hazard_label : str
        Label for the warming/hazard axis legend.
    title : str or None
        Plot title. Auto-generated if None.
    veg_labels : dict or None
        Mapping of integer class codes to label strings. Defaults to VEG_LABELS.
    sort_by : str
        One of 'refugia', 'at_risk', 'size', 'alpha'.
    figsize : tuple or None

    Returns
    -------
    fig, (ax, ax2, legend_ax)
    """
    if veg_labels is None:
        veg_labels = VEG_LABELS

    # Flatten and mask
    veg_flat = veg_aligned.flatten()
    bivar_flat = bivar_map.flatten()
    valid = np.isfinite(veg_flat) & (bivar_flat >= 0)

    veg_valid = veg_flat[valid].astype(int)
    bivar_valid = bivar_flat[valid].astype(int)

    # Cross-tabulate
    unique_veg = np.unique(veg_valid)
    n_veg = len(unique_veg)
    counts = np.zeros((n_veg, 9), dtype=int)
    for i, v in enumerate(unique_veg):
        mask_v = veg_valid == v
        for code in range(9):
            counts[i, code] = np.sum(bivar_valid[mask_v] == code)

    # Proportions (row-normalized)
    row_totals = counts.sum(axis=1, keepdims=True)
    proportions = np.where(row_totals > 0, 100.0 * counts / row_totals, 0)

    # Build label list
    cat_labels = [veg_labels.get(int(v), f"Class {int(v)}") for v in unique_veg]

    # Sort
    totals = counts.sum(axis=1)
    if sort_by == "refugia":
        sort_key = proportions[:, [0, 1, 2]].sum(axis=1)
        sort_idx = np.argsort(sort_key)[::-1]
    elif sort_by == "at_risk":
        sort_idx = np.argsort(proportions[:, 8])[::-1]
    elif sort_by == "size":
        sort_idx = np.argsort(totals)[::-1]
    elif sort_by == "alpha":
        sort_idx = np.argsort(cat_labels)
    else:
        sort_idx = np.arange(n_veg)

    sorted_props = proportions[sort_idx]
    sorted_totals = totals[sort_idx]
    sorted_labels = [cat_labels[i] for i in sort_idx]

    if figsize is None:
        figsize = (14, max(5, 0.45 * len(sorted_labels)))

    if title is None:
        title = f"Vegetation composition by nonant — {hazard_label}"

    # Plot
    fig, (ax, ax2) = plt.subplots(
        1, 2, figsize=figsize,
        gridspec_kw={"width_ratios": [4, 1], "wspace": 0.05},
        sharey=True,
    )

    y_pos = np.arange(len(sorted_labels))
    left = np.zeros(len(sorted_labels))

    for nonant in range(9):
        widths = sorted_props[:, nonant]
        ax.barh(y_pos, widths, left=left,
                color=bivar_colors[nonant],
                edgecolor="white", linewidth=0.5)
        left += widths

    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_labels, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of vegetation class")
    ax.set_title(title, fontsize=12, fontweight="bold", loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax2.barh(y_pos, sorted_totals, color="#555555")
    ax2.set_xlabel("Total pixels")
    ax2.set_xscale("log")
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.tick_params(labelsize=8)

    plt.subplots_adjust(right=0.82)

    # 3x3 legend
    legend_ax = fig.add_axes([0.86, 0.35, 0.12, 0.3])
    for i in range(3):
        for j in range(3):
            code = i * 3 + j
            legend_ax.add_patch(plt.Rectangle(
                (j, i), 1, 1,
                facecolor=bivar_colors[code],
                edgecolor="white", linewidth=1.5,
            ))
    legend_ax.set_xlim(-0.3, 3.3)
    legend_ax.set_ylim(-0.7, 3.5)
    legend_ax.set_aspect("equal")
    legend_ax.set_xticks([])
    legend_ax.set_yticks([])
    legend_ax.spines[:].set_visible(False)
    legend_ax.annotate("", xy=(3.15, 0), xytext=(0, 0),
                       arrowprops=dict(arrowstyle="->", color="black", lw=1.4))
    legend_ax.annotate("", xy=(0, 3.15), xytext=(0, 0),
                       arrowprops=dict(arrowstyle="->", color="black", lw=1.4))
    legend_ax.text(1.5, -0.6, "Conservation →", ha="center", fontsize=9)
    legend_ax.text(-0.5, 1.5, f"{hazard_label} →", ha="center", fontsize=9, rotation=90)

    return fig, (ax, ax2, legend_ax)

In [ ]:
VEG_PATH = (
    "/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/"
    "Vegetation_COWRA22/Vegetation_COWRA22.tif"
)

In [ ]:
LAYERS_TO_PLOT = ["5 day max precip", "drought__D3", "Days Exceeding 95th Percentile Maximum Temperature"]
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

PATH = FP_PATH  # swap this for any conservation/priority raster
OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/wp_bivariate")

# Build reference grid from the conservation raster
ref_transform, ref_width, ref_height = build_reference_grid(str(PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(PATH), ref_transform, ref_width, ref_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(ref_transform, ref_width, ref_height)

# Align vegetation to the reference grid (categorical → nearest)
veg_aligned = reproject_to_grid(
    str(VEG_PATH), ref_transform, ref_width, ref_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue
    for fp in files:
        clim_aligned = reproject_to_grid(
            fp, ref_transform, ref_width, ref_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        bivar_colors = layers[layer_name].get("bivar")
        hazard_label = cfg.get("label", layer_name)

        fig_veg, _ = plot_veg_composition(
            bivar_map=codes_2d,
            veg_aligned=veg_aligned,
            bivar_colors=bivar_colors,
            hazard_label=hazard_label,
            sort_by="size",
        )

        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        run_dir.mkdir(parents=True, exist_ok=True)
        fig_veg.savefig(run_dir / "veg_composition.png", dpi=150, bbox_inches="tight")

        # Clean up — order matters
        plt.close(fig_veg)
        plt.close('all')  # belt and suspenders, kills any stragglers
        del clim_aligned, cs_valid, clim_valid, mask_2d, codes, codes_2d, fig_veg
        gc.collect()

        print(f"✓ saved {run_dir.name}/veg_composition.png")